In [1]:
# 1_embedding_model_comparison_fixed.py

!pip install -q torch transformers sentence-transformers faiss-cpu pandas tqdm scikit-learn rouge-score nltk bert-score sacrebleu

import pandas as pd, numpy as np, torch, faiss, time, nltk, warnings, logging
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
from tqdm.auto import tqdm
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize
from bert_score import score as bert_score
from sacrebleu.metrics import BLEU

# Reduce HuggingFace model init warnings
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

# NLTK downloads
nltk.download("punkt", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

warnings.filterwarnings("ignore")

# ------------------- DATA -------------------
df = pd.read_csv("/kaggle/input/mlops-amazon/amazon.csv")

documents = [
    f"""Product: {r['product_name']}
Price: {r['discounted_price']} | Rating: {r['rating']} ({r['rating_count']} reviews)
Description: {r['about_product']}"""
    for _, r in df.iterrows()
]

TEST_QUERIES = [
    {
        "query": "Recommend a good fast charging USB-C cable under 300 rupees",
        "reference": "The boAt A400 at ₹299 is recommended, supporting 24W-25W fast charging and having a slightly higher customer rating. For a tighter budget, the pTron Solero TB301 at ₹149 supports 15W fast charging.",
    },
    {
        "query": "Which cable has the highest rating and supports 60W charging?",
        "reference": "The Belkin USB-C to USB-C Fast Charging Cable (60W PD) is the best match, offering a strong 4.5-star rating along with full 60W fast-charging support.",
    },
    {
        "query": "What is the best iPhone lightning cable in the list?",
        "reference": "For super-fast charging with a USB-C adapter, the Belkin Lightning to USB-C is best. For a reliable and durable standard cable with a good warranty, the Duracell USB-A to Lightning is a great choice. The Hi-Mobiler is the most budget-friendly option.",
    },
    {
        "query": "Suggest me some good long lasting headphones",
        "reference": "The boAt Bassheads 100 in-ear wired earphones are recommended for longevity, featuring a premium coated wire for sturdiness and a 1-year warranty. They have a 4.1-star rating from over 360,000 reviews and cost between ₹349-₹379.",
    },
]


# ------------------- METRICS CLASS -------------------
class Metrics:
    def __init__(self):
        self.rouge = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
        self.bleu = BLEU(effective_order=True)
        self.embedder = SentenceTransformer("all-MiniLM-L6-v2")

    def all(self, pred, ref, ctx):
        r = self.rouge.score(ref, pred)
        metrics = {
            "rouge_1_f1": r["rouge1"].fmeasure,
            "rouge_l_f1": r["rougeL"].fmeasure,
            "bleu": self.bleu.sentence_score(pred, [ref]).score / 100,
            "meteor": meteor_score([word_tokenize(ref.lower())], word_tokenize(pred.lower())),
        }
        # Use DeBERTa for BERTScore to avoid Roberta pooler warning
        P, R, F = bert_score(
            [pred], [ref], model_type="microsoft/deberta-large-mnli", verbose=False
        )
        metrics["bert_f1"] = F.mean().item()

        e1 = self.embedder.encode(pred)
        e2 = self.embedder.encode(ref)
        metrics["emb_sim"] = util.cos_sim(e1, e2).item()

        c_emb = self.embedder.encode(ctx)
        metrics["faith"] = min(1.0, 0.7 * util.cos_sim(e1, c_emb).item() + 0.3)

        return metrics

    def composite(self, m):
        w = {
            "rouge_1_f1": 0.1,
            "rouge_l_f1": 0.1,
            "bleu": 0.1,
            "meteor": 0.15,
            "bert_f1": 0.25,
            "emb_sim": 0.2,
            "faith": 0.1,
        }
        return sum(m[k] * w[k] for k in w)


metrics_calc = Metrics()


# ------------------- RAG CLASS -------------------
class RAG:
    def __init__(self, emb_name, generator):
        self.emb_name = emb_name
        self.generator = generator

        print(f"Loading embedding model: {emb_name}")
        self.embedder = SentenceTransformer(emb_name)

        dim = self.embedder.encode(["test"]).shape[1]
        self.index = faiss.IndexFlatIP(dim)

        print(f"Embedding {len(documents)} documents...")
        batches = [documents[i : i + 32] for i in range(0, len(documents), 32)]
        for b in tqdm(batches, desc="Indexing"):
            embs = self.embedder.encode(b, normalize_embeddings=True)
            self.index.add(embs)

    def retrieve(self, q, k):
        qe = self.embedder.encode([q], normalize_embeddings=True)
        D, I = self.index.search(qe, k)
        ctx = "\n\n".join([documents[i] for i in I[0]])
        return ctx

    def generate(self, q, ctx):
        prompt = f"Context:\n{ctx}\n\nQuestion: {q}\nAnswer:"
        out = self.generator(
            prompt,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.95,
            top_k=50,
            do_sample=True,
        )[0]["generated_text"]
        ans = out.split("Answer:")[-1].strip()
        return ans


# ------------------- LOAD GENERATOR ONCE -------------------
GEN_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"

print("\nLoading generator ONCE...")
generator = pipeline(
    "text-generation", model=GEN_MODEL, torch_dtype=torch.bfloat16, device_map="auto"
)

# ------------------- EXPERIMENT -------------------
results = []
EMBEDDING_MODELS = [
    "BAAI/bge-small-en-v1.5",
    "sentence-transformers/all-MiniLM-L6-v2",
    "BAAI/bge-base-en-v1.5",
]

for emb in EMBEDDING_MODELS:
    print(f"\n{'='*80}\nTESTING EMBEDDING: {emb}\n{'='*80}")
    rag = RAG(emb, generator)

    for qd in TEST_QUERIES:
        ctx = rag.retrieve(qd["query"], k=5)
        ans = rag.generate(qd["query"], ctx)
        m = metrics_calc.all(ans, qd["reference"], ctx)
        m["composite"] = metrics_calc.composite(m)

        results.append({**m, "embedding_model": emb, "query": qd["query"][:60]})
        print("\n------------------------------------------------------------")
        print(f"Embedding Model: {emb}")
        print(f"Query: {qd['query']}")
        print("\nRetrieved Context (first 500 chars):")
        print(ctx[:500] + "...")
        print("\nGenerated Answer:")
        print(ans)
        print("\nReference Answer:")
        print(qd["reference"])
        print(f"\nComposite Score: {m['composite']:.4f}")
        print("------------------------------------------------------------\n")


df_out = pd.DataFrame(results)
print("\n================ FINAL SUMMARY ================\n")

# Average composite score per embedding model
summary = df_out.groupby("embedding_model")["composite"].mean().sort_values(ascending=False)

print("Average Composite Scores:")
print(summary)

best_model = summary.idxmax()
best_score = summary.max()

print("\n---------------------------------------------")
print(f"🏆 Best Embedding Model: {best_model}")
print(f"🏅 Average Composite Score: {best_score:.4f}")
print("---------------------------------------------\n")

df_out.to_csv("1_embedding_comparison.csv", index=False)

print("\nEmbedding comparison saved → 1_embedding_comparison.csv")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6

2025-12-05 06:31:46.717657: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764916306.880081      21 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764916306.928360      21 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...


------------------------------------------------------------
Embedding Model: BAAI/bge-base-en-v1.5
Query: Which cable has the highest rating and supports 60W charging?

Retrieved Context (first 500 chars):
Product: MI Xiaomi USB Type C HYperCharge Cable 6A 100cm Sturdy and Durable Black Supports 120W HyperCharging
Price: ₹499 | Rating: 4.3 (30,411 reviews)
Description: Supports 120W Fast Charging|High Quality Design

Product: Zebronics CU3100V Fast charging Type C cable with QC 18W support, 3A max capacity, 1 meter braided cable, Data transfer and Superior durability (Braided Black + White)
Price: ₹139 | Rating: 3.9 (61 reviews)
Description: Fast charging support for various smart phones with the ...

Generated Answer:
The Belkin USB C to USB-C Fast Charging Type C Cable is the highest rated cable on the list, with a rating of 4.5 (577 reviews). It also supports 60W charging.

Reference Answer:
The Belkin USB-C to USB-C Fast Charging Cable (60W PD) is the best match, offering a stron

The following layers were not sharded: embeddings.LayerNorm.weight, encoder.layer.*.attention.self.v_bias, encoder.layer.*.intermediate.dense.bias, encoder.layer.*.output.dense.bias, encoder.layer.*.attention.self.in_proj.weight, encoder.layer.*.output.dense.weight, encoder.layer.*.attention.output.dense.weight, encoder.layer.*.attention.output.LayerNorm.weight, encoder.layer.*.attention.self.pos_proj.weight, encoder.layer.*.attention.output.LayerNorm.bias, encoder.layer.*.attention.self.pos_q_proj.bias, embeddings.LayerNorm.bias, encoder.layer.*.output.LayerNorm.bias, embeddings.word_embeddings.weight, encoder.layer.*.output.LayerNorm.weight, encoder.layer.*.intermediate.dense.weight, encoder.layer.*.attention.output.dense.bias, encoder.rel_embeddings.weight, encoder.layer.*.attention.self.q_bias, encoder.layer.*.attention.self.pos_q_proj.weight
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



------------------------------------------------------------
Embedding Model: BAAI/bge-base-en-v1.5
Query: What is the best iPhone lightning cable in the list?

Retrieved Context (first 500 chars):
Product: REDTECH USB-C to Lightning Cable 3.3FT, [Apple MFi Certified] Lightning to Type C Fast Charging Cord Compatible with iPhone 14/13/13 pro/Max/12/11/X/XS/XR/8, Supports Power Delivery - White
Price: ₹249 | Rating: 5.0 (nan reviews)
Description: 💎[The Fastest Charge] - This iPhone USB C cable supports PD 3.0 fast charging, up to 20W with USB-C Power Delivery adapters such as 18W, 20W, 29W, 30W, 61W, or 87W. Charge your iPhone from 0% to 50% in just 25 mins, and data transfer speeds up to 4...

Generated Answer:
It is difficult to determine the "best" iPhone lightning cable from the list as it depends on individual preferences and specific needs. However, some key features that could make a cable a good choice include fast charging capabilities, durability, and compatibility with vario

The following layers were not sharded: embeddings.LayerNorm.weight, encoder.layer.*.attention.self.v_bias, encoder.layer.*.intermediate.dense.bias, encoder.layer.*.output.dense.bias, encoder.layer.*.attention.self.in_proj.weight, encoder.layer.*.output.dense.weight, encoder.layer.*.attention.output.dense.weight, encoder.layer.*.attention.output.LayerNorm.weight, encoder.layer.*.attention.self.pos_proj.weight, encoder.layer.*.attention.output.LayerNorm.bias, encoder.layer.*.attention.self.pos_q_proj.bias, embeddings.LayerNorm.bias, encoder.layer.*.output.LayerNorm.bias, embeddings.word_embeddings.weight, encoder.layer.*.output.LayerNorm.weight, encoder.layer.*.intermediate.dense.weight, encoder.layer.*.attention.output.dense.bias, encoder.rel_embeddings.weight, encoder.layer.*.attention.self.q_bias, encoder.layer.*.attention.self.pos_q_proj.weight



------------------------------------------------------------
Embedding Model: BAAI/bge-base-en-v1.5
Query: Suggest me some good long lasting headphones

Retrieved Context (first 500 chars):
Product: Hp Wired On Ear Headphones With Mic With 3.5 Mm Drivers, In-Built Noise Cancelling, Foldable And Adjustable For Laptop/Pc/Office/Home/ 1 Year Warranty (B4B09Pa)
Price: ₹649 | Rating: 3.5 (7,222 reviews)
Description: Powerful bass and clear treble sounds|Wired connectivity|Ideal for long hours of listening|Superior sound quality and lengthy cable for easy of use|Compact and durable|The smart integrated in-cord remote facilitates easy audio control options|Additional Features: 35mm driver...

Generated Answer:
Based on your query, I have curated a list of long-lasting headphones that are suitable for your needs. These headphones offer superior sound quality, comfortable fit, and durable build.

1. Hp Wired On Ear Headphones With Mic - These wired on-ear headphones come with a 1-year warranty